In [1]:
import os
import json
from pathlib import Path

# =========================
# Config
# =========================
DATASET_ROOT = "."   # 例如: /workspace/mydata
OUTPUT_JSONL = "train.jsonl"

# 只处理这些类别文件夹
VALID_LABELS = {"good", "medium", "low"}

# 支持的图片后缀
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

PROMPT = (
    "This is a simulated image, not a real photograph. "
    "Please do not judge it too strictly by photo realism. "
    "Evaluate the semantic and spatial consistency of the image. "
    "Output only one label: good, medium, or low."
)


def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMG_EXTS


def find_label_from_path(path: Path, dataset_root: Path):
    """
    从相对路径的各级目录中寻找标签名。
    例如:
      good/a.jpg -> good
      some_dir/good/x.jpg -> good
    """
    rel_parts = path.relative_to(dataset_root).parts
    for p in rel_parts[:-1]:  # 文件名本身不参与判断
        if p.lower() in VALID_LABELS:
            return p.lower()
    return None


def build_sample(image_path: str, label: str):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": PROMPT}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": label}
                ]
            }
        ]
    }


def main():
    dataset_root = Path(DATASET_ROOT)
    if not dataset_root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    all_images = []
    for path in dataset_root.rglob("*"):
        if path.is_file() and is_image_file(path):
            label = find_label_from_path(path, dataset_root)
            if label is not None:
                all_images.append((path.resolve(), label))

    if len(all_images) == 0:
        print("No valid images found.")
        return

    label_count = {k: 0 for k in VALID_LABELS}

    with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
        for img_path, label in all_images:
            sample = build_sample(str(img_path), label)
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")
            label_count[label] += 1

    print(f"Done. Saved to: {OUTPUT_JSONL}")
    print(f"Total samples: {len(all_images)}")
    print("Label distribution:")
    for k in sorted(label_count.keys()):
        print(f"  {k}: {label_count[k]}")


if __name__ == "__main__":
    main()

Done. Saved to: train.jsonl
Total samples: 149
Label distribution:
  good: 31
  low: 38
  medium: 80


In [2]:
import json

TRAIN_JSONL = "train.jsonl"
VAL_JSONL = "val.jsonl"

OLD_PREFIX = "/workspace/download/OPA/new_OPA/composite"
NEW_PREFIX = "/data/gqs/qwen"


def replace_prefix_in_jsonl(input_path, output_path, old_prefix, new_prefix):
    with open(input_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    new_lines = []
    replaced_count = 0

    for line in lines:
        line = line.strip()
        if not line:
            continue

        sample = json.loads(line)

        if "messages" in sample:
            for msg in sample["messages"]:
                if "content" in msg:
                    for item in msg["content"]:
                        if item.get("type") == "image" and "image" in item:
                            old_path = item["image"]
                            if old_path.startswith(old_prefix):
                                item["image"] = old_path.replace(old_prefix, new_prefix, 1)
                                replaced_count += 1

        new_lines.append(json.dumps(sample, ensure_ascii=False))

    with open(output_path, "w", encoding="utf-8") as f:
        for line in new_lines:
            f.write(line + "\n")

    print(f"Done: {input_path} -> {output_path}")
    print(f"Replaced image path count: {replaced_count}")


if __name__ == "__main__":
    replace_prefix_in_jsonl(
        input_path=TRAIN_JSONL,
        output_path="train_replaced.jsonl",
        old_prefix=OLD_PREFIX,
        new_prefix=NEW_PREFIX
    )

    replace_prefix_in_jsonl(
        input_path=VAL_JSONL,
        output_path="val_replaced.jsonl",
        old_prefix=OLD_PREFIX,
        new_prefix=NEW_PREFIX
    )

Done: train.jsonl -> train_replaced.jsonl
Replaced image path count: 140
Done: val.jsonl -> val_replaced.jsonl
Replaced image path count: 9
